# Task 2: Data Preparation and Data Quality Assessment

This notebook inspects the source tables, validates their keys, enriches training observations with center and meal metadata, and records quality findings. It does not change files under `data/raw/`. Potential issues are retained and flagged when the available evidence cannot justify correction or exclusion. The test file is inspected for its schema only.

## 1. Load files and inspect schemas

The project root is located from the current working directory. Counts, inferred data types, missing values, exact duplicates, key uniqueness, and training business-key uniqueness are printed before merging.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((p for p in candidates if (p / 'data' / 'raw' / 'train.csv').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Could not locate data/raw/train.csv from the notebook working directory.')

raw_dir = project_root / 'data' / 'raw'
train_path = raw_dir / 'train.csv'
center_path = raw_dir / 'fulfilment_center_info.csv'
meal_path = raw_dir / 'meal_info.csv'
test_path = raw_dir / 'test.csv'
required_paths = [train_path, center_path, meal_path, test_path]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f'Missing required input files: {missing_paths}')

raw_hashes_before = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in required_paths}
train = pd.read_csv(train_path)
centers = pd.read_csv(center_path)
meals = pd.read_csv(meal_path)
test_schema = pd.read_csv(test_path, nrows=0)
test_sample = pd.read_csv(test_path, nrows=5)

def inspect_frame(name, frame, keys=()):
    print(f'\n{name}: rows={len(frame):,}, columns={len(frame.columns)}')
    print('Columns:', list(frame.columns))
    print('Data types:', {column: str(dtype) for column, dtype in frame.dtypes.items()})
    print('Missing values:', frame.isna().sum().to_dict())
    print('Exact duplicate rows:', int(frame.duplicated().sum()))
    for key in keys:
        if key in frame.columns:
            print(f'{key}: unique={frame[key].nunique(dropna=False):,}, duplicate rows={int(frame[key].duplicated(keep=False).sum()):,}, missing={int(frame[key].isna().sum()):,}')
    if 'num_orders' in frame.columns:
        print('num_orders total:', int(frame['num_orders'].sum()))

inspect_frame('train.csv', train, ['id'])
inspect_frame('fulfilment_center_info.csv', centers, ['center_id'])
inspect_frame('meal_info.csv', meals, ['meal_id'])
print('\ntest.csv schema only:')
print('Columns:', list(test_schema.columns))
print('Inferred types from up to five rows:', {column: str(dtype) for column, dtype in test_sample.dtypes.items()})

train_required = {'id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders'}
if not train_required.issubset(train.columns):
    raise ValueError(f'train.csv is missing required columns: {sorted(train_required - set(train.columns))}')
if not {'center_id'}.issubset(centers.columns) or not {'meal_id'}.issubset(meals.columns):
    raise ValueError('Dimension tables do not contain the expected join keys.')
business_key = ['week', 'center_id', 'meal_id']
print('\nTraining business-key duplicate rows:', int(train.duplicated(business_key, keep=False).sum()))
print('Training business-key duplicate groups:', int(train.loc[train.duplicated(business_key, keep=False), business_key].drop_duplicates().shape[0]))

key_unique = {'center_id': bool(centers['center_id'].is_unique), 'meal_id': bool(meals['meal_id'].is_unique)}
print('Dimension key uniqueness:', key_unique)
if not all(key_unique.values()):
    raise ValueError('A dimension join key is not unique; many-to-one merge validation is not supported.')

quality_rows = []
def add_quality(issue, detection, affected, denominator, decision, reason, impact):
    affected = int(affected)
    denominator = int(denominator)
    percentage = (100 * affected / denominator) if denominator else 0.0
    quality_rows.append({'issue': issue, 'detection': detection, 'affected': affected, 'denominator': denominator,
                         'percentage': percentage, 'decision': decision, 'reason': reason, 'impact': impact})

for name, frame in [('train.csv', train), ('fulfilment_center_info.csv', centers), ('meal_info.csv', meals)]:
    for column, count in frame.isna().sum().items():
        add_quality(f'Missing values: {name}.{column}', 'isna() by column', count, len(frame),
                    'Retain; no imputation or deletion. Flag if present.',
                    'Missingness must be understood before any value is inferred.',
                    'Could reduce usable observations or leave joined metadata unavailable.')
    add_quality(f'Exact duplicate rows: {name}', 'DataFrame.duplicated()', frame.duplicated().sum(), len(frame),
                'Retain and investigate if present; no automatic removal.',
                'A repeated row may be a valid observation or a data issue; confirm its meaning first.',
                'Duplicates can overweight records and inflate aggregates.')

business_duplicate_rows = int(train.duplicated(business_key, keep=False).sum())
business_duplicate_groups = int(train.loc[train.duplicated(business_key, keep=False), business_key].drop_duplicates().shape[0])
add_quality('Duplicate training business keys (week, center_id, meal_id)',
            'Rows belonging to repeated business-key groups; groups=' + str(business_duplicate_groups),
            business_duplicate_rows, len(train), 'Retain and investigate if present; no automatic removal.',
            'The combination should identify a weekly meal-center observation, but repetition alone does not establish which row is wrong.',
            'Repeated keys may make aggregation ambiguous or double-count demand.')

price_above = int(train['checkout_price'].gt(train['base_price']).sum())
add_quality('checkout_price > base_price', 'checkout_price greater than base_price', price_above, len(train),
            'Keep and flag; do not rewrite either price.',
            'The available fields do not establish whether the difference is a promotion, price change, or other business rule.',
            'Price comparisons should not be interpreted as discounts without additional context.')

q1, q3 = train['num_orders'].quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
order_outliers = train['num_orders'].lt(lower_fence) | train['num_orders'].gt(upper_fence)
add_quality('num_orders statistical outliers',
            f'IQR rule: Q1={q1:g}, Q3={q3:g}, IQR={iqr:g}, fences=[{lower_fence:g}, {upper_fence:g}]',
            order_outliers.sum(), len(train), 'Keep and flag; do not delete or cap.',
            'The IQR rule flags unusual values, not necessarily errors; no independent evidence justifies changing orders.',
            'Extreme demand observations can influence means and models; report robust summaries or sensitivity checks later.')
print(f'num_orders descriptive statistics:\n{train["num_orders"].describe()}')
print(f'IQR fences: lower={lower_fence:g}, upper={upper_fence:g}; low outliers={int(train["num_orders"].lt(lower_fence).sum()):,}; high outliers={int(train["num_orders"].gt(upper_fence).sum()):,}')

expected_weeks = pd.Index(range(int(train['week'].min()), int(train['week'].max()) + 1), name='week')
meal_ids = pd.Index(sorted(train['meal_id'].dropna().unique()), name='meal_id')
observed_meal_weeks = train[['meal_id', 'week']].drop_duplicates()
expected_pairs = pd.MultiIndex.from_product([meal_ids, expected_weeks], names=['meal_id', 'week'])
observed_pairs = pd.MultiIndex.from_frame(observed_meal_weeks)
missing_pairs = expected_pairs.difference(observed_pairs)
add_quality('Missing meal-week pairs in observed week range',
            f'Cartesian meal x contiguous observed week range {expected_weeks.min()}-{expected_weeks.max()}; missing pairs do not prove offer availability',
            len(missing_pairs), len(expected_pairs), 'Retain; flag as unobserved, not as zero demand.',
            'The files do not show whether a meal was offered in every week.',
            'Treating absent rows as zero would bias demand summaries if the meal was not offered.')
print(f'Missing meal-week pairs: {len(missing_pairs):,} / {len(expected_pairs):,}; missing meals are not assumed to have zero orders.')

numeric_columns = ['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders']
bad_type_rows = 0
nonfinite_mask = pd.Series(False, index=train.index)
for column in numeric_columns:
    parsed = pd.to_numeric(train[column], errors='coerce')
    bad_type_rows += int((parsed.isna() & train[column].notna()).sum())
    if is_numeric_dtype(parsed.dtype):
        nonfinite_mask |= ~np.isfinite(parsed)
nonfinite_rows = int(nonfinite_mask.sum())
add_quality('Non-numeric values in numeric training fields', 'to_numeric(errors=coerce), excluding existing nulls',
            bad_type_rows, len(train), 'Retain and flag if present; do not coerce silently.',
            'Automatic conversion could obscure malformed source values.',
            'Affected fields may not support numeric comparison or aggregation.')
add_quality('Non-finite values in numeric training fields', 'isfinite across numeric training fields',
            nonfinite_rows, len(train), 'Retain and investigate if present.',
            'Infinite values are not valid measurements.',
            'Can distort summaries and downstream calculations.')

invalid_checks = [
    ('Non-positive training id', train['id'].le(0).sum(), 'A record identifier should be positive.'),
    ('Non-positive week', train['week'].le(0).sum(), 'Observed week identifiers should be positive.'),
    ('Negative num_orders', train['num_orders'].lt(0).sum(), 'Negative order counts are not valid demand counts.'),
    ('Non-integer num_orders', train['num_orders'].mod(1).ne(0).sum(), 'Order counts are expected to be integer-valued.'),
    ('Non-positive checkout_price', train['checkout_price'].le(0).sum(), 'Prices are expected to be positive.'),
    ('Non-positive base_price', train['base_price'].le(0).sum(), 'Prices are expected to be positive.'),
    ('Invalid emailer_for_promotion flag', (~train['emailer_for_promotion'].isin([0, 1])).sum(), 'Promotion indicator is expected to be binary.'),
    ('Invalid homepage_featured flag', (~train['homepage_featured'].isin([0, 1])).sum(), 'Featured indicator is expected to be binary.'),
]
for issue, count, reason in invalid_checks:
    add_quality(issue, 'Domain validation', count, len(train), 'Retain and flag if present; no inferred correction.', reason,
                'Invalid values could affect identifiers, filters, or demand calculations.')

center_unknown = int((~train['center_id'].isin(centers['center_id'])).sum())
meal_unknown = int((~train['meal_id'].isin(meals['meal_id'])).sum())
add_quality('Training center keys absent from center dimension', 'train.center_id not in center_info.center_id',
            center_unknown, len(train), 'Preserve rows in left join; flag missing center metadata.',
            'Only keys present in the dimension can be enriched.', 'Missing attributes limit center-level analysis.')
add_quality('Training meal keys absent from meal dimension', 'train.meal_id not in meal_info.meal_id',
            meal_unknown, len(train), 'Preserve rows in left join; flag missing meal metadata.',
            'Only keys present in the dimension can be enriched.', 'Missing attributes limit meal-category analysis.')

def snapshot(label, frame):
    total = int(frame['num_orders'].sum()) if 'num_orders' in frame.columns else None
    print(f'{label}: rows={len(frame):,}, num_orders_sum={total:,}' if total is not None else f'{label}: rows={len(frame):,}, num_orders_sum=n/a')
    return len(frame), total

join_rows = []
prepared = train.copy()
initial_rows, initial_orders = snapshot('Before center merge', prepared)
unmatched_center_mask = ~prepared['center_id'].isin(centers['center_id'])
unmatched_center = int(unmatched_center_mask.sum())
unmatched_center_keys = int(prepared.loc[unmatched_center_mask, 'center_id'].nunique(dropna=False))
print(f'Center keys unmatched before merge: {unmatched_center_keys:,}; unmatched rows: {unmatched_center:,}')
before_rows, before_orders = len(prepared), int(prepared['num_orders'].sum())
prepared = prepared.merge(centers, on='center_id', how='left', validate='many_to_one', indicator='_center_merge')
center_rows, center_orders = snapshot('After center merge', prepared)
center_matched = int(prepared['_center_merge'].eq('both').sum())
center_unmatched = int(prepared['_center_merge'].eq('left_only').sum())
center_unmatched_keys = int(prepared.loc[prepared['_center_merge'].eq('left_only'), 'center_id'].nunique(dropna=False))
print(f'Center keys unmatched after merge: {center_unmatched_keys:,}; unmatched rows: {center_unmatched:,}')
join_rows.append({'issue': 'Center merge validation', 'detection': 'left many-to-one on center_id; matched rows=' + str(center_matched),
                  'affected': center_unmatched, 'denominator': before_rows, 'percentage': 100 * center_unmatched / before_rows,
                  'decision': 'Keep left-joined observations; inspect unmatched keys.',
                  'reason': f'Center key unique={key_unique["center_id"]}; source unmatched keys={unmatched_center_keys}, rows={unmatched_center}.',
                  'impact': f'Rows {before_rows:,}->{center_rows:,}; num_orders {before_orders:,}->{center_orders:,}; multiplication={max(0, center_rows-before_rows):,}; dropped={max(0, before_rows-center_rows):,}.'})
prepared = prepared.drop(columns='_center_merge')
assert center_rows == before_rows and center_orders == before_orders, 'Center left join changed observation count or num_orders total.'

before_rows, before_orders = len(prepared), int(prepared['num_orders'].sum())
unmatched_meal_mask = ~prepared['meal_id'].isin(meals['meal_id'])
unmatched_meal = int(unmatched_meal_mask.sum())
unmatched_meal_keys = int(prepared.loc[unmatched_meal_mask, 'meal_id'].nunique(dropna=False))
print(f'Meal keys unmatched before merge: {unmatched_meal_keys:,}; unmatched rows: {unmatched_meal:,}')
print(f'Before meal merge: rows={before_rows:,}, num_orders_sum={before_orders:,}')
prepared = prepared.merge(meals, on='meal_id', how='left', validate='many_to_one', indicator='_meal_merge')
meal_rows, meal_orders = snapshot('After meal merge', prepared)
meal_matched = int(prepared['_meal_merge'].eq('both').sum())
meal_unmatched = int(prepared['_meal_merge'].eq('left_only').sum())
meal_unmatched_keys = int(prepared.loc[prepared['_meal_merge'].eq('left_only'), 'meal_id'].nunique(dropna=False))
print(f'Meal keys unmatched after merge: {meal_unmatched_keys:,}; unmatched rows: {meal_unmatched:,}')
join_rows.append({'issue': 'Meal merge validation', 'detection': 'left many-to-one on meal_id; matched rows=' + str(meal_matched),
                  'affected': meal_unmatched, 'denominator': before_rows, 'percentage': 100 * meal_unmatched / before_rows,
                  'decision': 'Keep left-joined observations; inspect unmatched keys.',
                  'reason': f'Meal key unique={key_unique["meal_id"]}; source unmatched keys={unmatched_meal_keys}, rows={unmatched_meal}.',
                  'impact': f'Rows {before_rows:,}->{meal_rows:,}; num_orders {before_orders:,}->{meal_orders:,}; multiplication={max(0, meal_rows-before_rows):,}; dropped={max(0, before_rows-meal_rows):,}.'})
prepared = prepared.drop(columns='_meal_merge')
assert meal_rows == before_rows and meal_orders == before_orders, 'Meal left join changed observation count or num_orders total.'

final_rows, final_orders = snapshot('Final prepared dataset', prepared)
comparison = pd.DataFrame([
    {'stage': 'Original training data', 'rows': initial_rows, 'num_orders_sum': initial_orders},
    {'stage': 'After center and meal joins', 'rows': final_rows, 'num_orders_sum': final_orders},
])
print('\nOriginal versus merged integrity:')
display(comparison)
assert final_rows == initial_rows, 'Final merge did not preserve training observation count.'
assert final_orders == initial_orders, 'Final merge did not preserve the training num_orders total.'
print('Join integrity confirmed: row count and total num_orders are preserved.')

quality = pd.DataFrame(quality_rows + join_rows)
print('\nQuality check summary (each percentage uses its stated denominator):')
display(quality[['issue', 'detection', 'affected', 'denominator', 'percentage', 'decision']])

processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
prepared_path = processed_dir / 'prepared_train.csv'
prepared.to_csv(prepared_path, index=False)
reloaded = pd.read_csv(prepared_path)
assert len(reloaded) == final_rows, 'Reloaded prepared dataset has a different row count.'
assert int(reloaded['num_orders'].sum()) == final_orders, 'Reloaded prepared dataset has a different num_orders total.'
print(f'Prepared dataset saved and reloaded successfully: {prepared_path.relative_to(project_root)}')

def fmt_number(value):
    return f'{int(value):,}'

def fmt_percent(value):
    return f'{value:.2f}%'

unresolved = quality.loc[quality['affected'].gt(0), ['issue', 'affected', 'percentage', 'decision']]
print('\nUnresolved or flagged issues (retained, not silently removed):')
if unresolved.empty:
    print('No nonzero quality findings were detected by the checks in this notebook.')
else:
    display(unresolved)

arabic_headers = ['المشكلة', 'طريقة الكشف', 'عدد السجلات المتأثرة', 'النسبة', 'القرار', 'السبب', 'الأثر / القيود']
decision_lines = [
    '# قرارات تجهيز البيانات وجودتها',
    '',
    'أُنشئ هذا الجدول من نتائج تشغيل `notebooks/data_preparation.ipynb`. لم تُعدّل ملفات `data/raw/`، ولم تُحذف أو تُصحح سجلات من التدريب. النسب محسوبة من المقام الموضح في فحص كل مشكلة.',
    '',
    '| ' + ' | '.join(arabic_headers) + ' |',
    '| ' + ' | '.join(['---'] * len(arabic_headers)) + ' |',
]
for row in quality.to_dict(orient='records'):
    cells = [str(row['issue']), str(row['detection']), fmt_number(row['affected']) + ' / ' + fmt_number(row['denominator']),
             fmt_percent(row['percentage']), str(row['decision']), str(row['reason']), str(row['impact'])]
    decision_lines.append('| ' + ' | '.join(value.replace('|', '\\|').replace('\n', ' ') for value in cells) + ' |')
decision_lines.extend([
    '',
    '## سلامة الدمج',
    '',
    f'- الصفوف: {initial_rows:,} قبل الدمج و{final_rows:,} بعده.',
    f'- مجموع `num_orders`: {initial_orders:,} قبل الدمج و{final_orders:,} بعده.',
    f'- ملف البيانات المجهزة: `data/processed/prepared_train.csv` ({final_rows:,} صفًا).',
    '- بقيت ملاحظات الجودة غير المحسومة معلّمة في الدفتر والجدول؛ غياب صف وجبة-أسبوع لا يثبت أن الوجبة كانت معروضة أو أن طلبها صفر.',
    '',
])
decisions_path = project_root / 'docs' / 'decisions_cleaning.md'
decisions_path.parent.mkdir(parents=True, exist_ok=True)
decisions_path.write_text('\n'.join(decision_lines), encoding='utf-8')
print(f'Cleaning decisions documented from measured results: {decisions_path.relative_to(project_root)}')

raw_hashes_after = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in required_paths}
assert raw_hashes_after == raw_hashes_before, 'A raw input file changed during notebook execution.'
print('Raw input integrity confirmed: all source file SHA-256 hashes are unchanged.')



train.csv: rows=456,548, columns=9
Columns: ['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders']
Data types: {'id': 'int64', 'week': 'int64', 'center_id': 'int64', 'meal_id': 'int64', 'checkout_price': 'float64', 'base_price': 'float64', 'emailer_for_promotion': 'int64', 'homepage_featured': 'int64', 'num_orders': 'int64'}
Missing values: {'id': 0, 'week': 0, 'center_id': 0, 'meal_id': 0, 'checkout_price': 0, 'base_price': 0, 'emailer_for_promotion': 0, 'homepage_featured': 0, 'num_orders': 0}
Exact duplicate rows: 0


id: unique=456,548, duplicate rows=0, missing=0
num_orders total: 119557485

fulfilment_center_info.csv: rows=77, columns=5
Columns: ['center_id', 'city_code', 'region_code', 'center_type', 'op_area']
Data types: {'center_id': 'int64', 'city_code': 'int64', 'region_code': 'int64', 'center_type': 'str', 'op_area': 'float64'}
Missing values: {'center_id': 0, 'city_code': 0, 'region_code': 0, 'center_type': 0, 'op_area': 0}
Exact duplicate rows: 0
center_id: unique=77, duplicate rows=0, missing=0

meal_info.csv: rows=51, columns=3
Columns: ['meal_id', 'category', 'cuisine']
Data types: {'meal_id': 'int64', 'category': 'str', 'cuisine': 'str'}
Missing values: {'meal_id': 0, 'category': 0, 'cuisine': 0}
Exact duplicate rows: 0
meal_id: unique=51, duplicate rows=0, missing=0

test.csv schema only:
Columns: ['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured']
Inferred types from up to five rows: {'id': 'int64', 'week': 'int64', '

num_orders descriptive statistics:
count    456548.000000
mean        261.872760
std         395.922798
min          13.000000
25%          54.000000
50%         136.000000
75%         324.000000
max       24299.000000
Name: num_orders, dtype: float64
IQR fences: lower=-351, upper=729; low outliers=0; high outliers=32,937
Missing meal-week pairs: 222 / 7,395; missing meals are not assumed to have zero orders.
Before center merge: rows=456,548, num_orders_sum=119,557,485
Center keys unmatched before merge: 0; unmatched rows: 0
After center merge: rows=456,548, num_orders_sum=119,557,485
Center keys unmatched after merge: 0; unmatched rows: 0
Meal keys unmatched before merge: 0; unmatched rows: 0
Before meal merge: rows=456,548, num_orders_sum=119,557,485


After meal merge: rows=456,548, num_orders_sum=119,557,485
Meal keys unmatched after merge: 0; unmatched rows: 0
Final prepared dataset: rows=456,548, num_orders_sum=119,557,485

Original versus merged integrity:


,stage,rows,num_orders_sum
0,Original training data,456548,119557485
1,After center and meal joins,456548,119557485


Join integrity confirmed: row count and total num_orders are preserved.

Quality check summary (each percentage uses its stated denominator):


,issue,detection,affected,denominator,percentage,decision
0,Missing values: train.csv.id,isna() by column,0,456548,0.000000,Retain; no imputation or deletion. Flag if pre...
1,Missing values: train.csv.week,isna() by column,0,456548,0.000000,Retain; no imputation or deletion. Flag if pre...
2,Missing values: train.csv.center_id,isna() by column,0,456548,0.000000,Retain; no imputation or deletion. Flag if pre...
3,Missing values: train.csv.meal_id,isna() by column,0,456548,0.000000,Retain; no imputation or deletion. Flag if pre...
4,Missing values: train.csv.checkout_price,isna() by column,0,456548,0.000000,Retain; no imputation or deletion. Flag if pre...
5,Missing values: train.csv.base_price,isna() by column,0,456548,0.000000,Retain; no imputation or deletion. Flag if pre...
6,Missing values: train.csv.emailer_for_promotion,isna() by column,0,456548,0.000000,Retain; no imputation or deletion. Flag if pre...
7,Missing values: train.csv.homepage_featured,isna() by column,0,456548,0.000000,Retain; no imputation or deletion. Flag if pre...
8,Missing values: train.csv.num_orders,isna() by column,0,456548,0.000000,Retain; no imputation or deletion. Flag if pre...
9,Exact duplicate rows: train.csv,DataFrame.duplicated(),0,456548,0.000000,Retain and investigate if present; no automati...


Prepared dataset saved and reloaded successfully: data\processed\prepared_train.csv

Unresolved or flagged issues (retained, not silently removed):


,issue,affected,percentage,decision
21,checkout_price > base_price,116101,25.430185,Keep and flag; do not rewrite either price.
22,num_orders statistical outliers,32937,7.214356,Keep and flag; do not delete or cap.
23,Missing meal-week pairs in observed week range,222,3.002028,"Retain; flag as unobserved, not as zero demand."


Cleaning decisions documented from measured results: docs\decisions_cleaning.md
Raw input integrity confirmed: all source file SHA-256 hashes are unchanged.


## Interpretation and handling

All quality findings are flags rather than automatic deletion rules. Exact duplicate rows and repeated business keys are reported separately. IQR outliers are statistical observations, not proven errors. Missing meal-week combinations are compared with the contiguous observed week range but cannot be classified as missed records versus weeks with no offer from these files alone. The generated cleaning log records the measured count, percentage, decision, rationale, and limitation for each check, including both merge validations.